# Partition Cell Types Deepclean

Perform deepcleaning removal of doublet clusters based on manual annotation

Relabel L3 AIFI cell type labels based on deepcleaning manual annotations

## Load libraries

In [17]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc
import scanpy.external as sce
import glob


In [2]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

## Helper functions

These make it a bit simpler to cache and read in files from HISE

In [3]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [4]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

In [5]:
def sort_adata_uuid(uuid, sort_cols = ['AIFI_L2', 'sample.sampleKitGuid']):
    cache_file = cache_uuid_path(uuid)
    adata = sc.read_h5ad(cache_file)
    obs = adata.obs
    obs = obs.sort_values(sort_cols)
    adata = adata[obs.index]
    adata.write_h5ad(cache_file)

This function will enable us to connect to our .h5ad files without loading the entire thing into memory. We'll then load only the cells that we want for each cell class to assemble them for writing. This should save us some overhead as we do our subsetting.

In [6]:
def read_adata_backed_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = sc.read_h5ad(cache_file, backed = 'r')
    return res

This function will remove Ig-related genes, which is recommended by Marla Glass for analysis of B cell subtypes

In [7]:
def remove_ig_genes(adata):
    igl_genes = [gene for gene in adata.var_names if gene.startswith("IGL")]
    igk_genes = [gene for gene in adata.var_names if gene.startswith("IGK")]
    ighc_genes = [gene for gene in adata.var_names if gene.startswith("IGH")]
    exl_genes = igl_genes + igk_genes + ighc_genes

    filtered_genes = [gene for gene in adata.var_names if gene not in exl_genes]
    adata = adata[:, filtered_genes]

    return adata

This function will apply a standard normalization, nearest neighbors, clustering, and UMAP process to our cell subsets:

In [8]:
def process_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    adata = adata.raw.to_adata()
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')
    
    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_{r}'.format(r = resolution),
        n_iterations = 2
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05)
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [9]:
def format_cell_type(cell_type):
    cell_type = re.sub('\\+', 'pos', cell_type)
    cell_type = re.sub('-', 'neg', cell_type)
    cell_type = re.sub(' ', '_', cell_type)
    return cell_type

In [10]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

In [11]:
# make a function to find files
def get_filepaths_with_glob(root_path: str, file_regex: str):
    return glob.glob(os.path.join(root_path, file_regex))

In [177]:
# Define a function to extract the desired substring using regex
def extract_substring(path, pattern = r'preRA_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'):
    #pattern = r'preRA_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'
    match = re.search(pattern, path)
    if match:
        return match.group(1)
    return None

In [13]:
def read_anndata_files(file_tuples):
    """
    Read Anndata objects from H5AD files and store them in a dictionary with custom names.

    Parameters:
        file_tuples (list of tuples): List of tuples where each tuple contains filename and desired name.

    Returns:
        dict: Dictionary containing Anndata objects with custom names.
    """
    anndata_dict = {}
    for filename, name in file_tuples:
        anndata_obj = anndata.read_h5ad(filename)
        anndata_dict[name] = anndata_obj
    return anndata_dict

In [14]:
def reformat_cell_type(cell_type):
    '''convert cell type names read in from file back to original L3 labels'''
    cell_type = re.sub('pos', '+', cell_type)
    cell_type = re.sub('neg', '-', cell_type)
    cell_type = re.sub('_',' ', cell_type)
    return cell_type

## Identify files for use in HISE

In [15]:
search_id = 'polonium-strontium-zirconium'

Retrieve files stored in our HISE project store

In [18]:
ps_df = hisepy.list_files_in_project_store('UCSDCU_Y4')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [19]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

In [20]:
h5ad_df = search_df[search_df['name'].str.contains('.h5ad')]

In [21]:
h5ad_df

,id,name
410,704874eb-d661-4ca3-bb4c-a0f4c20c5fe9,polonium-strontium-zirconium/preRA_cluster_har...
411,3ccbd155-2b80-4fd7-8988-ef4462198ccf,polonium-strontium-zirconium/preRA_cluster_har...
412,2db8c682-b9da-4be7-87ce-4449618222e8,polonium-strontium-zirconium/preRA_cluster_har...
413,3d754896-d258-449e-9739-f6d921607201,polonium-strontium-zirconium/preRA_cluster_har...
414,00a560a8-499c-4cc3-98fc-b4f93f6f0d91,polonium-strontium-zirconium/preRA_cluster_har...
...,...,...
476,e7ae8e4e-c9f5-4798-9a7a-7fbd7f6593e2,polonium-strontium-zirconium/preRA_cluster_har...
477,40e4d947-9308-4ada-8681-e62f66a37077,polonium-strontium-zirconium/preRA_cluster_har...
478,b01b7837-cb0b-4c57-ba62-e133943e9852,polonium-strontium-zirconium/preRA_cluster_har...
479,b75ca414-cb11-402f-aea5-8d1e000a2dad,polonium-strontium-zirconium/preRA_cluster_har...


In [22]:
h5ad_uuids = {}
for i in range(h5ad_df.shape[0]):
    fn = h5ad_df['name'].tolist()[i]
    group_name = re.sub('.+pbmc_', '', fn)
    group_name = re.sub('_init.+', '', group_name)
    h5ad_uuids[group_name] = h5ad_df['id'].tolist()[i]

In [23]:
h5ad_uuids

{'polonium-strontium-zirconium/preRA_cluster_harmonize_ASDC_2024-05-25.h5ad': '704874eb-d661-4ca3-bb4c-a0f4c20c5fe9',
 'polonium-strontium-zirconium/preRA_cluster_harmonize_Activated_memory_B_cell_2024-05-25.h5ad': '3ccbd155-2b80-4fd7-8988-ef4462198ccf',
 'polonium-strontium-zirconium/preRA_cluster_harmonize_Adaptive_NK_cell_2024-05-25.h5ad': '2db8c682-b9da-4be7-87ce-4449618222e8',
 'polonium-strontium-zirconium/preRA_cluster_harmonize_BaEoMaP_cell_2024-05-25.h5ad': '3d754896-d258-449e-9739-f6d921607201',
 'polonium-strontium-zirconium/preRA_cluster_harmonize_C1Qpos_CD16_monocyte_2024-05-25.h5ad': '00a560a8-499c-4cc3-98fc-b4f93f6f0d91',
 'polonium-strontium-zirconium/preRA_cluster_harmonize_CD14pos_cDC2_2024-05-25.h5ad': '63217d12-5f51-4668-9ad8-dea66b394581',
 'polonium-strontium-zirconium/preRA_cluster_harmonize_CD27neg_effector_B_cell_2024-05-25.h5ad': 'f9d59339-9182-48e0-abf2-8a4d48cee5f6',
 'polonium-strontium-zirconium/preRA_cluster_harmonize_CD27pos_effector_B_cell_2024-05-25.h5

In [24]:
len(h5ad_uuids)

71

## Download and sort .h5ad files

Sorting the .h5ad files by AIFI_L2 will make reading each cell type much faster by placing cells of the same type next to each other in the sparse matrix.

In [ ]:

# run once
for uuid in h5ad_uuids.values():
    sort_adata_uuid(uuid, sort_cols = ['AIFI_L3'])

downloading fileID: 704874eb-d661-4ca3-bb4c-a0f4c20c5fe9
Files have been successfully downloaded!
downloading fileID: 3ccbd155-2b80-4fd7-8988-ef4462198ccf
Files have been successfully downloaded!
downloading fileID: 2db8c682-b9da-4be7-87ce-4449618222e8
Files have been successfully downloaded!
downloading fileID: 3d754896-d258-449e-9739-f6d921607201
Files have been successfully downloaded!
downloading fileID: 00a560a8-499c-4cc3-98fc-b4f93f6f0d91
Files have been successfully downloaded!
downloading fileID: 63217d12-5f51-4668-9ad8-dea66b394581
Files have been successfully downloaded!
downloading fileID: f9d59339-9182-48e0-abf2-8a4d48cee5f6
Files have been successfully downloaded!
downloading fileID: abd08fae-e2ed-4a09-8c2e-38c2531c56eb
Files have been successfully downloaded!
downloading fileID: 6edd318c-5481-4b7c-99a6-f4cdcd44daba
Files have been successfully downloaded!
downloading fileID: 68a4eda0-8312-4f07-a661-c834dcc403f7
Files have been successfully downloaded!
downloading fileID: 

## Open connections to .h5ad files

Now that they're sorted, we can open these files with on-disk backing so we don't have to read the entire file at once.

## Process files

In [25]:
input_path = "../../05-clustering/scripts/output/"

In [26]:
### read in processed leiden adata
filenames = get_filepaths_with_glob(input_path, "preRA_cluster_harmonize_*.h5ad")  
filenames[:5]

['../../05-clustering/scripts/output/preRA_cluster_harmonize_CD4_MAIT_2024-05-25.h5ad',
 '../../05-clustering/scripts/output/preRA_cluster_harmonize_ISGpos_naive_CD8_T_cell_2024-05-26.h5ad',
 '../../05-clustering/scripts/output/preRA_cluster_harmonize_Adaptive_NK_cell_2024-05-25.h5ad',
 '../../05-clustering/scripts/output/preRA_cluster_harmonize_Early_memory_B_cell_2024-05-26.h5ad',
 '../../05-clustering/scripts/output/preRA_cluster_harmonize_CD56bright_NK_cell_2024-05-25.h5ad']

In [27]:
### extract cell types
# Apply the function to each filename in the list using list comprehension
cell_types = [extract_substring(path) for path in filenames]
cell_types[:5]

['CD4_MAIT',
 'ISGpos_naive_CD8_T_cell',
 'Adaptive_NK_cell',
 'Early_memory_B_cell',
 'CD56bright_NK_cell']

In [28]:
file_dict = dict(zip(cell_types, filenames))
list(file_dict.items())[:10]

[('CD4_MAIT',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_CD4_MAIT_2024-05-25.h5ad'),
 ('ISGpos_naive_CD8_T_cell',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_ISGpos_naive_CD8_T_cell_2024-05-26.h5ad'),
 ('Adaptive_NK_cell',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_Adaptive_NK_cell_2024-05-25.h5ad'),
 ('Early_memory_B_cell',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_Early_memory_B_cell_2024-05-26.h5ad'),
 ('CD56bright_NK_cell',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_CD56bright_NK_cell_2024-05-25.h5ad'),
 ('Intermediate_monocyte',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_Intermediate_monocyte_2024-05-26.h5ad'),
 ('Core_naive_CD8_T_cell',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_Core_naive_CD8_T_cell_2024-05-26.h5ad'),
 ('Transitional_B_cell',
  '../../05-clustering/scripts/output/preRA_cluster_harmonize_Transitional_B_cell_2024-05-26.h5ad'),
 ('pDC

In [86]:
#### read in doublet csv
doublet_df = pd.read_csv("../data/consensus_doublet_removal_sheet_06_12_24.csv")
### select only cell type and doublet leiden
doublet_df = doublet_df[["AIFI_L3", "leiden (doublet clusters to be removed)"]]
# reformat cell types
doublet_df['AIFI_L3'] = [format_cell_type(cell_type) for cell_type in doublet_df['AIFI_L3']]
doublet_df

,AIFI_L3,leiden (doublet clusters to be removed)
0,ASDC,"2,11,16"
1,Activated_memory_B_cell,"2,12"
2,Adaptive_NK_cell,"9,18,22,24"
3,BaEoMaP_cell,7
4,C1Qpos_CD16_monocyte,"16,17,18,19,21,24,25"
...,...,...
66,SOX4pos_naive_CD8_T_cell,"10,21"
67,Transitional_B_cell,"14,15,18"
68,Type_2_polarized_memory_B_cell,"11,18,20"
69,cDC1,"5,18,19,20,23"


In [87]:
doublet_df['AIFI_L3']

0                               ASDC
1            Activated_memory_B_cell
2                   Adaptive_NK_cell
3                       BaEoMaP_cell
4               C1Qpos_CD16_monocyte
                   ...              
66          SOX4pos_naive_CD8_T_cell
67               Transitional_B_cell
68    Type_2_polarized_memory_B_cell
69                              cDC1
70                               pDC
Name: AIFI_L3, Length: 71, dtype: object

## Store doublet clusters

In [88]:
# Convert the DataFrame to a dictionary
doublet_cluster_dict = {}

for index, row in doublet_df.iterrows():
    key = row['AIFI_L3']
    # Check if the value is not NaN or None
    if pd.notna(row['leiden (doublet clusters to be removed)']):
        values = row['leiden (doublet clusters to be removed)'].split(',')
    else:
        values = []
    doublet_cluster_dict[key] = values

#print(doublet_cluster_dict)

In [89]:
doublet_cluster_dict

{'ASDC': ['2', '11', '16'],
 'Activated_memory_B_cell': ['2', '12'],
 'Adaptive_NK_cell': ['9', '18', '22', '24'],
 'BaEoMaP_cell': ['7'],
 'C1Qpos_CD16_monocyte': ['16', '17', '18', '19', '21', '24', '25'],
 'CD14pos_cDC2': ['4', '16', '18', '19', '21', '22'],
 'CD27pos_effector_B_cell': ['15', ' 20', '19', '22'],
 'CD27neg_effector_B_cell': ['13', '18', '21'],
 'CD4_MAIT': ['14'],
 'CD56bright_NK_cell': ['10', '21', '22', '19'],
 'CD8_MAIT': ['14', '15', '18', '19', '22', '23'],
 'CD8aa': ['18'],
 'CD95_memory_B_cell': ['7', '8', '16'],
 'CLP_cell': ['11'],
 'CM_CD4_T_cell': [],
 'CM_CD8_T_cell': ['18', '19'],
 'CMP_cell': ['15', '16', '20', '21'],
 'Core_CD14_monocyte': ['16', '17', '22', '24', '28'],
 'Core_CD16_monocyte': ['13', '16', '18', '20', '21'],
 'Core_memory_B_cell': ['14', '16', '18', '19', '20'],
 'Core_naive_B_cell': ['8', '11', '13', '18', '19', '20'],
 'Core_naive_CD4_T_cell': ['15', '17', '19', '20', ' 21'],
 'Core_naive_CD8_T_cell': ['13'],
 'DN_T_cell': ['22', '23

## Store Relabel Cell Type Clusters

Store list of leiden clusters to relabel to L3 cell types

In [162]:
relabel_df = pd.read_csv("../data/relabel_deepclean_sheet_06_11_24.csv")
relabel_df = relabel_df[['AIFI_L3', 'relabel_name','clusters']]
# reformat cell types
relabel_df['AIFI_L3'] = [format_cell_type(cell_type) for cell_type in relabel_df['AIFI_L3']]
# filter by cell types that need to be relabeled and clusters only
relabel_df = relabel_df.dropna(thresh= 2)
relabel_df

,AIFI_L3,relabel_name,clusters
0,ASDC,ASDC_uk1_B,"0,1,7"
1,Activated_memory_B_cell,Activated memory B cell_uk1,1
2,Adaptive_NK_cell,Adaptive NK cell_uk1_T,"7,11,20,21"
14,CM_CD4_T_cell,"CM CD4 T cell_uk1_CD8,CM CD4 T cell_uk2","12,17"
24,Early_memory_B_cell,Early memory B cell_uk1,"12,13"
27,GZMBneg_CD27pos_EM_CD4_T_cell,GZMB- CD27+ EM CD4 T cell_uk1_CD8,15
28,GZMBneg_CD27neg_EM_CD4_T_cell,GZMB- CD27- EM CD4 T cell_uk1_CD8,6
29,GZMKpos_CD27pos_EM_CD8_T_cell,GZMK+ CD27+ EM CD8 T cell_uk1_gdt,"10,15,17"
37,ILC,ILC_uk1,2
43,ISGpos_memory_CD4_T_cell,ISG+ memory CD4 T cell_uk1_CD8,15


In [98]:
### remove cell types with more than one names to be relabeled
multiple_relabel_cond = relabel_df['AIFI_L3'].str.contains('|'.join(['CM_CD4_T_cell', 'KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell']))
relabel_df_single = relabel_df[~multiple_relabel_cond]
relabel_df_multi = relabel_df[multiple_relabel_cond]

In [99]:
relabel_df_single

,AIFI_L3,relabel_name,clusters
0,ASDC,ASDC_uk1_B,"0,1,7"
1,Activated_memory_B_cell,Activated memory B cell_uk1,1
2,Adaptive_NK_cell,Adaptive NK cell_uk1_T,"7,11,20,21"
24,Early_memory_B_cell,Early memory B cell_uk1,"12,13"
27,GZMBneg_CD27pos_EM_CD4_T_cell,GZMB- CD27+ EM CD4 T cell_uk1_CD8,15
28,GZMBneg_CD27neg_EM_CD4_T_cell,GZMB- CD27- EM CD4 T cell_uk1_CD8,6
29,GZMKpos_CD27pos_EM_CD8_T_cell,GZMK+ CD27+ EM CD8 T cell_uk1_gdt,"10,15,17"
37,ILC,ILC_uk1,2
43,ISGpos_memory_CD4_T_cell,ISG+ memory CD4 T cell_uk1_CD8,15
53,KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell,KLRF1-_GZMB+_CD27-_EM_CD8_uk1,20


In [100]:
relabel_df_multi

,AIFI_L3,relabel_name,clusters
14,CM_CD4_T_cell,"CM CD4 T cell_uk1_CD8,CM CD4 T cell_uk2","12,17"
51,KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell,"[KLRF1+_GZMB+_CD27-_EM_CD8_uk1],[KLRF1+_GZMB+_...","[30],[23],[1,4,15,16]"


In [83]:
# Create the dictionary
relabel_cluster_dict = {}

for index, row in relabel_df_single.iterrows():
    key = row['AIFI_L3']
    # parse clusters that have multiple relabel name within same cell types
    cl = row['clusters'].split(',')
    parsed_cl =  [cluster.strip('[]').split(',') for cluster in cl]
    value = (row['relabel_name'].split(','), cl)
    relabel_cluster_dict[key] = value



In [84]:
print(relabel_cluster_dict)

{'ASDC': (['ASDC_uk1_B'], ['0', '1', '7']), 'Activated_memory_B_cell': (['Activated memory B cell_uk1'], ['1']), 'Adaptive_NK_cell': (['Adaptive NK cell_uk1_T'], ['7', '11', '20', '21']), 'Early_memory_B_cell': (['Early memory B cell_uk1'], ['12', '13']), 'GZMBneg_CD27pos_EM_CD4_T_cell': (['GZMB- CD27+ EM CD4 T cell_uk1_CD8'], ['15']), 'GZMBneg_CD27neg_EM_CD4_T_cell': (['GZMB- CD27- EM CD4 T cell_uk1_CD8'], ['6']), 'GZMKpos_CD27pos_EM_CD8_T_cell': (['GZMK+ CD27+ EM CD8 T cell_uk1_gdt'], ['10', '15', '17']), 'ILC': (['ILC_uk1'], ['2']), 'ISGpos_memory_CD4_T_cell': (['ISG+ memory CD4 T cell_uk1_CD8'], ['15']), 'KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell': (['KLRF1-_GZMB+_CD27-_EM_CD8_uk1'], ['20']), 'KLRF1neg_GZMBpos_CD27neg_memory_CD4_T_cell': (['KLRF1-_GZMB+_CD27-_mem_CD4_uk1'], ['5', '6']), 'Proliferating_T_cell': (['Prolif_T_uk1'], ['16']), 'Type_2_polarized_memory_B_cell': (['T2MBC_uk1'], ['19'])}


In [150]:
### manually add cell types with multiple renaming labels to dictionary
multi_relabel_dict = {
    'KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell': (
        ['KLRF1+_GZMB+_CD27-_EM_CD8_uk1', 'KLRF1+_GZMB+_CD27-_EM_CD8_uk2', 'KLRF1+GZMB+_CD27-_EM_CD8_uk3'],
        [['30'], ['23'], ['1','4','15','16']]),
    'CM_CD4_T_cell': (
        ['CM CD4 T cell_uk1_CD8', 'CM CD4 T cell_uk2'], 
        [['12'],['17']])}
multi_relabel_dict

{'KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell': (['KLRF1+_GZMB+_CD27-_EM_CD8_uk1',
   'KLRF1+_GZMB+_CD27-_EM_CD8_uk2',
   'KLRF1+GZMB+_CD27-_EM_CD8_uk3'],
  [['30'], ['23'], ['1', '4', '15', '16']]),
 'CM_CD4_T_cell': (['CM CD4 T cell_uk1_CD8', 'CM CD4 T cell_uk2'],
  [['12'], ['17']])}

## Process Each Cell Types

In [122]:
### create unit test: cell type with no renaming, cell type with 1 rename and cell type with multiple rename
#subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'ASDC', 'CM_CD4_T_cell'))
subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'ASDC', 'CM_CD4_T_cell'))

subfile_dict

{'CD4_MAIT': '../../05-clustering/scripts/output/preRA_cluster_harmonize_CD4_MAIT_2024-05-25.h5ad',
 'ASDC': '../../05-clustering/scripts/output/preRA_cluster_harmonize_ASDC_2024-05-25.h5ad',
 'CM_CD4_T_cell': '../../05-clustering/scripts/output/preRA_cluster_harmonize_CM_CD4_T_cell_2024-05-26.h5ad'}

In [163]:
out_files = []
out_files

[]

In [164]:
for label, file in file_dict.items():
    print("Key:", label)
    print("Value:", file)

    adata = sc.read_h5ad(file)
    print(adata)

    # subset doublet dictionary by current cell types
    doublet_clust = doublet_cluster_dict[label]
    print('Doublet clusters: ' + str(doublet_clust))

    #### label doublets on metadata
    #### create new column in metadata that labels doublets based on target leiden from manual analysis 
    if len(doublet_clust) == 0:
        print("The list is empty")
        adata.obs['doublets_manual'] = 'no'
    else:
        adata.obs['doublets_manual'] = ['yes' if leiden in doublet_clust else 'no' for leiden in adata.obs['leiden_harmony_2']]
    ## check
    print(pd.crosstab(adata.obs['doublets_manual'], adata.obs['leiden_harmony_2']))


    #### label renamed cell types
    # if current cell type needs to be relabeled and the current cell type does not have multiple relabel names    
    if label in relabel_df_single['AIFI_L3'].values and label not in relabel_df_multi['AIFI_L3'].values:
        relabel_clust = relabel_cluster_dict[label]

        print('Clusters to relabel: '+ str(relabel_clust))
        to_rename = relabel_clust[0][0]
        rename_cl = relabel_clust[1]
        # modify metadata
        adata.obs['AIFI_L3_new'] = [
            to_rename if leiden in rename_cl else adata.obs['AIFI_L3'][idx] 
            for idx, leiden in enumerate(adata.obs['leiden_harmony_2'])
        ]
        # check
        print(pd.crosstab(adata.obs['AIFI_L3_new'], adata.obs['leiden_harmony_2']))
    
    # if curr cell type contains multiple cell types to relabel
    if label in relabel_df_multi['AIFI_L3'].values:
        relabel_clust = multi_relabel_dict[label]
    
        print('Clusters to relabel: '+ str(relabel_clust))
    
        # Create a dictionary from the relabel_clust for easier lookup
        rename_dict = {}
        for cell_type, clusters in zip(relabel_clust[0], relabel_clust[1]):
            for cluster in clusters:
                rename_dict[cluster] = cell_type
    
        # Apply the renaming logic
        adata.obs['AIFI_L3_new'] = [
            rename_dict.get(leiden, adata.obs['AIFI_L3'][idx]) 
            for idx, leiden in enumerate(adata.obs['leiden_harmony_2'])
        ]
        print(pd.crosstab(adata.obs['AIFI_L3_new'], adata.obs['leiden_harmony_2']))

    # if curr cell type does not need to be relabeled, create new column, inherit old L3 annotations
    if label not in  relabel_df_single['AIFI_L3'].values and label not in relabel_df_multi['AIFI_L3'].values:
        adata.obs['AIFI_L3_new'] = adata.obs['AIFI_L3']

    ### export doublet metadata, do not append to out-files for certpro
    meta = adata.obs[['barcodes','batch_id', 'cell_name', 'sample.sampleKitGuid','subject.subjectGuid','AIFI_L3_new','leiden_harmony_2','doublets_manual']]
    meta.head()
    ### export
    meta.to_csv('../data/preRA_doublet_meta_{c}_{d}.csv'.format(
        c = label,
        d = date.today()
    ))

    # subset anndata by singlets only
    adata_subset = adata[adata.obs['doublets_manual'] == 'no']

    print("subsetted cells to: " + str(adata.shape))

    # export deep cleaned anndata
    print('Saving processed data')
    out_file = 'output/preRA_deepcleaned_{c}_{d}.h5ad'.format(
        c = label,
        d = date.today()
    )
    adata_subset.write_h5ad(out_file)

    # append to outfile list
    out_files.append(out_file)

Key: CD4_MAIT
Value: ../../05-clustering/scripts/output/preRA_cluster_harmonize_CD4_MAIT_2024-05-25.h5ad
AnnData object with n_obs × n_vars = 4270 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: ISGpos_naive_CD8_T_cell
Value: ../../05-clustering/scripts/output/preRA_cluster_harmonize_ISGpos_naive_CD8_T_cell_2024-05-26.h5ad
AnnData object with n_obs × n_vars = 2299 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: Adaptive_NK_cell
Value: ../../05-clustering/scripts/output/preRA_cluster_harmonize_Adaptive_NK_cell_2024-05-25.h5ad
AnnData object with n_obs × n_vars = 73124 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'tot

...

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: CD95_memory_B_cell
Value: ../../05-clustering/scripts/output/preRA_cluster_harmonize_CD95_memory_B_cell_2024-05-25.h5ad
AnnData object with n_obs × n_vars = 7540 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', '

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: GZMBneg_CD27pos_EM_CD4_T_cell
Value: ../../05-clustering/scripts/output/preRA_cluster_harmonize_GZMBneg_CD27pos_EM_CD4_T_cell_2024-05-26.h5ad
AnnData object with n_obs × n_vars = 326393 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_cou

/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1292: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


In [165]:
out_files

['output/preRA_deepcleaned_CD4_MAIT_2024-06-13.h5ad',
 'output/preRA_deepcleaned_ISGpos_naive_CD8_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Adaptive_NK_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Early_memory_B_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_CD56bright_NK_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Intermediate_monocyte_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Core_naive_CD8_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Transitional_B_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_pDC_2024-06-13.h5ad',
 'output/preRA_deepcleaned_KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_CM_CD4_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Core_naive_B_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Memory_CD8_Treg_2024-06-13.h5ad',
 'output/preRA_deepcleaned_CD8aa_2024-06-13.h5ad',
 'output/preRA_deepcleaned_ISGpos_MAIT_2024-06-13.h5ad',
 'output/preRA_deepcleaned_ILC_2024-06-13.h5ad',
 'output/preRA

In [166]:
len(out_files)

71

## Export Doublet Metadata

In [39]:
doublet_meta_comb = pd.concat(doublet_meta)

In [42]:
### check metadata
len(doublet_meta_comb['AIFI_L3'].unique())

71

In [43]:
### export
doublet_meta_comb.to_csv("../data/preRA_scRNA_512_samples_doublet_metadata_all_types.csv")

## Upload Cell Type data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [173]:
study_space_uuid = '223de760-9624-45bd-aefe-ca24c75b1800'
title = 'IDE for 06 PBMC L3 Deepclean {d}'.format(d = date.today())
title

'IDE for 06 PBMC L3 Deepclean 2024-06-13'

In [168]:
search_id = element_id()
search_id

'molybdenum-niobium-krypton'

In [169]:
in_files = list(h5ad_uuids.values())
in_files

['704874eb-d661-4ca3-bb4c-a0f4c20c5fe9',
 '3ccbd155-2b80-4fd7-8988-ef4462198ccf',
 '2db8c682-b9da-4be7-87ce-4449618222e8',
 '3d754896-d258-449e-9739-f6d921607201',
 '00a560a8-499c-4cc3-98fc-b4f93f6f0d91',
 '63217d12-5f51-4668-9ad8-dea66b394581',
 'f9d59339-9182-48e0-abf2-8a4d48cee5f6',
 'abd08fae-e2ed-4a09-8c2e-38c2531c56eb',
 '6edd318c-5481-4b7c-99a6-f4cdcd44daba',
 '68a4eda0-8312-4f07-a661-c834dcc403f7',
 'fe28ae5f-8f35-49ec-a1d8-6a5723c1d80f',
 '38d11eaf-6f15-4908-8b23-90990b55ec98',
 '5a22953e-96a2-4b38-b79f-edddaf98ad39',
 '8c9ccdaa-30c5-4afa-ae8a-9d88e008ef18',
 '920734fd-943d-42b2-8216-7fc458e30c77',
 '23088aa5-7ec2-4e2a-a117-a83c2c0485d7',
 '69fd7a3a-b26e-4cb3-b3cd-76584c7f25ea',
 '2d91757a-efcb-4428-b396-0d82abebbbfa',
 '8083ad79-29cb-4838-81b6-9481b348b8b7',
 '2e5e1d81-855d-4fc7-ba30-d5a69904ba4d',
 '80d25082-5f70-4f4e-9ef1-c9d0373c7061',
 '6a34dd0b-d984-4397-a049-92b3782fb327',
 '8d68c740-389d-4e68-bda8-aad6dde3ee10',
 '0eeb81ba-3caa-4d76-95b8-05346f4a405b',
 'e8439aa9-7a54-

In [170]:
out_files

['output/preRA_deepcleaned_CD4_MAIT_2024-06-13.h5ad',
 'output/preRA_deepcleaned_ISGpos_naive_CD8_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Adaptive_NK_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Early_memory_B_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_CD56bright_NK_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Intermediate_monocyte_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Core_naive_CD8_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Transitional_B_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_pDC_2024-06-13.h5ad',
 'output/preRA_deepcleaned_KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_CM_CD4_T_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Core_naive_B_cell_2024-06-13.h5ad',
 'output/preRA_deepcleaned_Memory_CD8_Treg_2024-06-13.h5ad',
 'output/preRA_deepcleaned_CD8aa_2024-06-13.h5ad',
 'output/preRA_deepcleaned_ISGpos_MAIT_2024-06-13.h5ad',
 'output/preRA_deepcleaned_ILC_2024-06-13.h5ad',
 'output/preRA

In [171]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/preRA_deepcleaned_CD4_MAIT_2024-06-13.h5ad', 'output/preRA_deepcleaned_ISGpos_naive_CD8_T_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_Adaptive_NK_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_Early_memory_B_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_CD56bright_NK_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_Intermediate_monocyte_2024-06-13.h5ad', 'output/preRA_deepcleaned_Core_naive_CD8_T_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_Transitional_B_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_pDC_2024-06-13.h5ad', 'output/preRA_deepcleaned_KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_CM_CD4_T_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_Core_naive_B_cell_2024-06-13.h5ad', 'output/preRA_deepcleaned_Memory_CD8_Treg_2024-06-13.h5ad', 'output/preRA_deepcleaned_CD8aa_2024-06-13.h5ad', 'output/preRA_deepcleaned_ISGpos_MAIT_2024-06-13.h5ad', 'output/preRA_deepcleaned_ILC_2024-06-13

(y/n) y


{'trace_id': 'eed482fd-9d4e-4752-8d90-b908296eff6f',
 'files': ['output/preRA_deepcleaned_CD4_MAIT_2024-06-13.h5ad',
  'output/preRA_deepcleaned_ISGpos_naive_CD8_T_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_Adaptive_NK_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_Early_memory_B_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_CD56bright_NK_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_Intermediate_monocyte_2024-06-13.h5ad',
  'output/preRA_deepcleaned_Core_naive_CD8_T_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_Transitional_B_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_pDC_2024-06-13.h5ad',
  'output/preRA_deepcleaned_KLRF1pos_GZMBpos_CD27neg_EM_CD8_T_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_CM_CD4_T_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_Core_naive_B_cell_2024-06-13.h5ad',
  'output/preRA_deepcleaned_Memory_CD8_Treg_2024-06-13.h5ad',
  'output/preRA_deepcleaned_CD8aa_2024-06-13.h5ad',
  'output/preRA_deepcleaned_ISGpos_MAIT_2024

In [172]:
import session_info
session_info.show()